In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_13_try_1.pickle

In [4]:
%%RecordEvent
%%time
### cell 14 ###

### cell 14 (optimized for cudf)

# define the question strings
question_of_interest_cell26 = (
    'On which platforms have you begun or completed data science courses?'
)
question_of_interest_alternate = (
    'On which online platforms have you begun or completed data science courses'
)

# helper to grab and rename only the response columns for a given question
def grab_subset_of_data_26(original_df, question_of_interest):
    subset_cols = [c for c in original_df.columns if question_of_interest in c]
    mapping = {c: c.split('-')[-1].lstrip() for c in subset_cols}
    # select, rename, and drop rows where all of these columns are null
    return (
        original_df[subset_cols]
                   .rename(columns=mapping)
                   .dropna(how='all')
    )

# combine multiple years into one DataFrame and also produce per-year counts
def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest,
        include_2017=False):
    years = ['2018', '2019', '2020', '2021', '2022']
    sources = [responses_df_2018_cell10,
               responses_df_2019_cell10,
               responses_df_2020,
               responses_df_2021,
               responses_df_2022_cell10]
    if include_2017:
        years.insert(0, '2017')
        sources.insert(0, responses_df_2017)

    dfs = []
    for src_df, yr in zip(sources, years):
        sub = grab_subset_of_data_26(src_df, question_of_interest)
        sub['year'] = yr
        dfs.append(sub)

    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts

# convert the raw counts into percentages per year
def convert_df_of_counts_to_percentages_26(df_combined, df_counts):
    # compute total respondents per year
    totals = df_combined['year'].value_counts().sort_index()
    # divide each count by the corresponding total and scale to 100
    pct = df_counts.set_index('year').div(totals, axis=0) * 100
    return pct.reset_index()

# apply the original in-place column and value replacements
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(
            question_of_interest_alternate,
            question_of_interest_cell26,
            regex=False
        )
)
responses_df_2018_cell10.replace(['Kaggle Learn'], 'Kaggle Learn Courses', inplace=True)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace('Kaggle Learn', 'Kaggle Learn Courses', regex=False)
)
responses_df_2018_cell10.replace(['Fast.AI'], 'Fast.ai', inplace=True)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace('Fast.AI', 'Fast.ai', regex=False)
)
responses_df_2018_cell10.replace(
    ['Online University Courses'],
    'University Courses (resulting in a university degree)',
    inplace=True
)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(
            'Online University Courses',
            'University Courses (resulting in a university degree)',
            regex=False
        )
)
responses_df_2019_cell10.replace(
    ['Kaggle Courses (i.e. Kaggle Learn)'],
    'Kaggle Learn Courses',
    inplace=True
)
responses_df_2019_cell10.columns = (
    responses_df_2019_cell10.columns
        .str.replace(
            'Kaggle Courses (i.e. Kaggle Learn)',
            'Kaggle Learn Courses',
            regex=False
        )
)

# combine, count, convert to percentages, then reshape for plotting
(learning_platform_df_combined,
 learning_platform_df_combined_counts) = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest_cell26,
        include_2017=False
    )

learning_platform_df_combined_percentages = \
    convert_df_of_counts_to_percentages_26(
        learning_platform_df_combined,
        learning_platform_df_combined_counts
    )

# clean up the college-course label and select desired column order
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
        .str.replace(
            '(resulting in a university degree)',
            '',
            regex=False
        )
)
keep = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
learning_platform_df_combined_percentages_cell26 = \
    learning_platform_df_combined_percentages.loc[:, keep]

# melt into long form and sort

df = (
    learning_platform_df_combined_percentages_cell26
        .melt(id_vars=['year'], value_vars=[c for c in keep if c != 'year'])
)
df_cell26 = (
    df.sort_values(['year', 'value'], ascending=True, ignore_index=True)
)
df_cell26.columns = df_cell26.columns.str.replace('variable', '', regex=False)

df_cell26.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    50 non-null     object
 1           50 non-null     object
 2   value   50 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.4+ KB
CPU times: user 2.26 s, sys: 76.1 ms, total: 2.34 s
Wall time: 2.33 s


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_14_try_3.pickle